#### Wavelet Synchrosqueezed Transform (WSST)

In [16]:
import os
import numpy as np
import matplotlib.pyplot as plt
from obspy import read
from ssqueezepy import ssq_cwt

In [17]:
# -------- CONFIGURATION --------
input_folder = "C:/work/all_paper/papers/p3/Final submission/sr/final_editted/plot_EQ/"       
output_folder = "C:/work/all_paper/papers/p3/Final submission/sr/final_editted/plot_EQ/" 
os.makedirs(output_folder, exist_ok=True)

# WSST frequency settings
f_c = 0.8125            # Morlet center frequency
fmin, fmax = 1, 30      # Displayed frequency range (Hz)
nfreqs = 200  

In [ ]:
# -------- LOOP THROUGH FILES --------
for filename in os.listdir(input_folder):
    if filename.endswith(".mseed"):
        file_path = os.path.join(input_folder, filename)

        try:
            
            st = read(file_path)
            tr = st[0]
            data = tr.data.astype(float)
            fs = tr.stats.sampling_rate
            npts = tr.stats.npts
            times = np.arange(npts) / fs
            data = (data - np.mean(data)) / np.std(data)

            freqs = np.logspace(np.log10(fmin), np.log10(fmax), nfreqs)
            scales = f_c * fs / freqs
            scales = np.sort(scales)

            Tx, _, _, _ = ssq_cwt(data, fs=fs, wavelet='morlet', scales=scales)
            amplitude = np.abs(Tx)
            actual_freqs = f_c * fs / scales  # Map scales to Hz

            amplitude = np.flipud(amplitude)
            actual_freqs = actual_freqs[::-1]

            plt.figure(figsize=(10, 6))
            im = plt.imshow(amplitude, aspect='auto', origin='lower',
                            extent=[times[0], times[-1], actual_freqs[0], actual_freqs[-1]],
                            cmap='turbo', vmin=0, vmax=np.percentile(amplitude, 99))
            plt.colorbar(im, label='Amplitude')
            plt.xlabel('Time (s)')
            plt.ylabel('Frequency (Hz)')
            plt.title(f'WSST Heatmap: {tr.id} | {tr.stats.starttime.isoformat()} — {tr.stats.endtime.isoformat()}')
            plt.ylim(fmin, fmax)
            plt.tight_layout()

            save_path = os.path.join(output_folder, f"{os.path.splitext(filename)[0]}_wsst_rich1.png")
            plt.savefig(save_path, dpi=450)
            plt.close()
            print(f"Saved: {save_path}")

        except Exception as e:
            print(f"Error processing {filename}: {e}")
